# ⚙️ Feature Engineering: Tabular Data (Jurnal)
Tujuan dari notebook ini adalah melakukan transformasi data mentah menjadi format yang siap diolah oleh model *Machine Learning*. Langkah yang dilakukan meliputi pembuatan fitur IMT, *Encoding*, *Standardization*, dan *Splitting* dataset.

## Import Library

In [9]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Path setup
DATA_INPUT_PATH = '../data/processed/dataset_stunting_clean.csv'
DATA_OUTPUT_DIR = '../data/data_fe'

print("✅ Library & Path terkonfigurasi.")

✅ Library & Path terkonfigurasi.


## Load Dataset

In [10]:
print("⏳ Memuat dataset...")
df = pd.read_csv(DATA_INPUT_PATH)
df.head()

⏳ Memuat dataset...


,id_anak,jenis_kelamin,usia_bulan,berat_badan,tinggi_badan,status_bb_u,zscore_bb_u,status_tb_u,zscore_tb_u,status_bb_tb,zscore_bb_tb
0,1,Perempuan,54.0,13.2,97.5,Normal,-1.94,Stunting,-2.11,Normal,-0.95
1,2,Laki-laki,44.0,12.0,92.0,Normal,-1.92,Stunting,-2.22,Normal,-0.88
2,3,Laki-laki,57.0,14.0,97.0,Normal,-1.90,Stunting,-2.58,Normal,-0.48
3,4,Laki-laki,26.0,11.0,79.0,Normal,-1.15,Stunting,-3.11,Normal,0.68
4,5,Perempuan,59.0,14.6,98.0,Normal,-1.66,Stunting,-2.49,Normal,-0.18


## Feature Creation (IMT & Rasio Tinggi/Usia)

In [11]:
print("⚙️ Menghitung fitur rekayasa antropometri...")

# 1. Rumus IMT: Berat (kg) / (Tinggi (m) ^ 2)
df['imt'] = df['berat_badan'] / ((df['tinggi_badan'] / 100) ** 2)

# 2. Rumus Rasio Tinggi terhadap Usia (cm/bulan)
# Kita tambahkan +1 pada penyebut untuk menghindari pembagian dengan angka 0 (Zero Division) jika ada bayi usia 0 bulan
df['tinggi_per_usia'] = df['tinggi_badan'] / (df['usia_bulan'] + 1)

print("✅ Berhasil menambah fitur 'imt' dan 'tinggi_per_usia'.")
print("\n--- Sampel Hasil Rekayasa Fitur ---")
df[['usia_bulan', 'tinggi_badan', 'berat_badan', 'imt', 'tinggi_per_usia']].head()

⚙️ Menghitung fitur rekayasa antropometri...
✅ Berhasil menambah fitur 'imt' dan 'tinggi_per_usia'.

--- Sampel Hasil Rekayasa Fitur ---


,usia_bulan,tinggi_badan,berat_badan,imt,tinggi_per_usia
0,54.0,97.5,13.2,13.885602,1.772727
1,44.0,92.0,12.0,14.177694,2.044444
2,57.0,97.0,14.0,14.879371,1.672414
3,26.0,79.0,11.0,17.625381,2.925926
4,59.0,98.0,14.6,15.201999,1.633333


## Feature Encoding

In [12]:
print("⚙️ Mengubah variabel kategorikal menjadi numerik...")

le_jk = LabelEncoder()
df['jenis_kelamin_encoded'] = le_jk.fit_transform(df['jenis_kelamin'])

le_target = LabelEncoder()
df['status_tb_u_encoded'] = le_target.fit_transform(df['status_tb_u'])

print(f"Mapping JK: {dict(zip(le_jk.classes_, le_jk.transform(le_jk.classes_)))}")
print(f"Mapping Status: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")

⚙️ Mengubah variabel kategorikal menjadi numerik...
Mapping JK: {'Laki-laki': np.int64(0), 'Perempuan': np.int64(1)}
Mapping Status: {'Stunting': np.int64(0), 'Tidak Stunting': np.int64(1)}


## Train-Test Split

In [13]:
# Daftarkan semua fitur termasuk fitur rasio yang baru dibuat
fitur_pilihan = ['usia_bulan', 'jenis_kelamin_encoded', 'tinggi_badan', 'berat_badan', 'imt', 'tinggi_per_usia']
X = df[fitur_pilihan]
y = df['status_tb_u_encoded']

# Split 80:20 dengan mempertahankan proporsi kelas target (stratify)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"✅ Split selesai dengan {len(fitur_pilihan)} fitur:")
print(f"   -> Train set: {X_train.shape[0]} baris")
print(f"   -> Test set : {X_test.shape[0]} baris")

✅ Split selesai dengan 6 fitur:
   -> Train set: 32052 baris
   -> Test set : 8014 baris


## Feature Scaling

In [14]:
print("⚙️ Scaling data dengan StandardScaler...")
scaler = StandardScaler()

# Fit & Transform hanya di train untuk menghindari Data Leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Kembalikan ke format DataFrame dengan kolom fitur terbaru
X_train_final = pd.DataFrame(X_train_scaled, columns=fitur_pilihan)
X_test_final = pd.DataFrame(X_test_scaled, columns=fitur_pilihan)

print("✅ Scaling seluruh fitur selesai.")
X_train_final.head()

⚙️ Scaling data dengan StandardScaler...
✅ Scaling seluruh fitur selesai.


,usia_bulan,jenis_kelamin_encoded,tinggi_badan,berat_badan,imt,tinggi_per_usia
0,0.943634,-0.905538,0.500198,-0.023101,-0.028364,-0.431894
1,0.247905,-0.905538,0.161364,0.019624,-0.005371,-0.323860
2,1.512867,1.104316,1.262575,0.179842,-0.000229,-0.465463
3,1.639363,1.104316,1.516700,0.126436,-0.019552,-0.467446
4,-0.384576,-0.905538,0.025830,-0.012420,-0.010513,-0.134058


## Save Final Features

In [15]:
os.makedirs(DATA_OUTPUT_DIR, exist_ok=True)

# Gabung kembali target untuk disimpan
X_train_final['target'] = y_train.values
X_test_final['target'] = y_test.values

X_train_final.to_csv(os.path.join(DATA_OUTPUT_DIR, 'tabular_train_features.csv'), index=False)
X_test_final.to_csv(os.path.join(DATA_OUTPUT_DIR, 'tabular_test_features.csv'), index=False)

print("🏁 SUCCESS: File fitur siap untuk Modeling!")

🏁 SUCCESS: File fitur siap untuk Modeling!
